# EVCharge Intelligence — Member 2 temporal analysis

Analyze session-start demand and recorded energy using the verified Phase 2 tables. This notebook validates the cleaned data through existing functions, checks the saved summaries, and imports reusable plotting functions. It does not implement station/queue analysis or Power BI.

Methodological motivation: Baek, K., Lee, E., & Kim, J. (2024), [A dataset for multi-faceted analysis of electric vehicle charging transactions](https://doi.org/10.1038/s41597-024-02942-9), *Scientific Data*, 11, 262. The paper motivates examining charging transactions over time. Its dataset and numerical results are separate from ours. Session starts differ from connection/coupling profiles; calendar normalization and the relative demand index are project choices.

Source status: **shared by team member; original source and licence unverified**. Methodology and timezone remain unverified.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
ROOT = current if (current / "src/demand_analysis.py").is_file() else current.parent
if not (ROOT / "src/demand_analysis.py").is_file():
    raise RuntimeError("Open from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_preparation import file_hash
from src.demand_analysis import analyze_sessions
from src.demand_visualization import load_analysis, generate_figures

summary_dir = ROOT / "outputs/demand"
protected = [ROOT / "data/raw/EV_Dataset.csv", ROOT / "data/processed/charging_sessions.csv",
             *sorted(summary_dir.glob("*.csv")), summary_dir / "analysis_summary.json"]
before = {path: file_hash(path) for path in protected}
tables, summary = load_analysis(summary_dir)
cleaned = pd.read_csv(ROOT / "data/processed/charging_sessions.csv", dtype=str, keep_default_na=False)
recomputed, checked_summary = analyze_sessions(cleaned)
for name, saved in tables.items():
    pd.testing.assert_frame_equal(saved, recomputed[name], check_dtype=False, rtol=1e-10, atol=1e-6)
assert checked_summary["reconciliation"]["all_passed"]
del cleaned, recomputed
print(f"Validated {summary['input_session_count']:,} sessions; all saved summaries agree with existing analysis functions.")

Validated 200,000 sessions; all saved summaries agree with existing analysis functions.


## Calendar denominators and recorded-energy eligibility

The inclusive range is 1 January 2024 through 30 December 2025 (730 dates). Complete collection over that range, including zero-session dates, is an assumption, not verified uptime; first/last days may be partially observed. December 2025 represents 30 of 31 dates and is marked `*`; this is calendar truncation, not a missing-data marker.

Small denominator example: if a range includes two Fridays and one Saturday, three Friday starts give 3/2 = 1.5 starts per Friday; two Saturday starts give 2/1 = 2 starts per Saturday. Comparing raw totals would obscure that difference. Weekday–hour denominators use that weekday's dates, while each hour uses all represented dates. Year-months stay separate.

Energy is numerically valid when finite and >= 0. Zero is allowed; missing, nonnumeric, nonfinite or negative energy is excluded only from energy metrics, never demand. A plausibility flag is separate: energy above battery capacity or nominal charger power × elapsed hours is a concern under unverified field definitions, not proof of an invalid reading. Baseline includes all numerically valid energy.

In [2]:
display(pd.DataFrame([{
    "sessions": summary["input_session_count"],
    "calendar_days": summary["coverage"]["inclusive_calendar_days"],
    "sessions_per_day": summary["avg_sessions_per_day"],
    "valid_energy_sessions": summary["valid_energy_session_count"],
    "invalid_energy_sessions": summary["invalid_energy_session_count"],
    "recorded_energy_kwh": summary["total_recorded_energy_kwh"],
}]).round(2))
display(tables["weekday_weekend_demand"][["Day_Type", "session_count", "calendar_days", "avg_sessions_per_day"]].round(2))
display(tables["monthly_demand"].loc[[0, 3, 15, 23], ["Year_Month", "calendar_days", "avg_sessions_per_day", "is_partial_boundary_month"]].round(2))
paths = generate_figures(tables, summary_dir / "figures")
print(f"Saved {len(paths)} figure files (PNG 300 dpi and SVG). Figures below link to these artifacts without embedded image duplicates.")

,sessions,calendar_days,sessions_per_day,valid_energy_sessions,invalid_energy_sessions,recorded_energy_kwh
0,200000,730,273.97,200000,0,9493219.61


,Day_Type,session_count,calendar_days,avg_sessions_per_day
0,Weekday,143330,522,274.58
1,Weekend,56670,208,272.45


,Year_Month,calendar_days,avg_sessions_per_day,is_partial_boundary_month
0,2024-01,31,273.16,False
3,2024-04,30,270.00,False
15,2025-04,30,278.83,False
23,2025-12,30,281.13,True


Saved 14 figure files (PNG 300 dpi and SVG). Figures below link to these artifacts without embedded image duplicates.


## Hourly session starts

![Hourly session starts](../outputs/demand/figures/01_hourly_demand.png)

Hourly demand is **nearly uniform**: relative index 0.97668–1.01760. Hour 08 has 11.62 starts per calendar day (8,480 total), and hour 09 has 11.15 (8,139). The dashed mean is 11.42 starts per hour per day. These small descriptive differences do not establish congestion or statistical significance.

## Daily starts and smoothing

![Daily session starts](../outputs/demand/figures/02_daily_demand.png)

The busiest observed date is 26 January 2024 (344 starts); the lowest is 11 October 2025 (230). The trailing 7-day mean requires seven observations, so the first six values are missing. No cause or sustained growth is inferred from the day-to-day variation.

## Weekday and weekend demand

![Weekday comparisons](../outputs/demand/figures/03_weekday_demand.png)

Weekdays average 274.58 starts/day across 522 dates; weekends average 272.45 across 208 dates. Wednesday is highest at 276.83 per represented Wednesday; Sunday is lowest at 271.14. The weekday/weekend difference is 2.13 starts/day, about 0.78% of the weekend rate; this is descriptive, not a significance test.

## Weekday–hour pattern

![Weekday–hour heatmap](../outputs/demand/figures/04_day_hour_heatmap.png)

Wednesday 02 has the highest weekday–hour rate (12.27); Tuesday 09 has the lowest (10.52). The sequential color scale starts at zero so small differences are not visually overstated. A higher session-start rate does not show simultaneous connected vehicles or congestion.

## Monthly demand

![Monthly demand](../outputs/demand/figures/05_monthly_demand.png)

Average starts/day range from 270.00 in April 2024 to 281.13 in December 2025. The latter covers only 1–30 December, not a full month. Among full calendar months, April 2025 is highest at 278.83/day. The figures do not establish a seasonal cause or statistically significant trend.

## Recorded energy associated with session-start periods

![Recorded-energy profiles](../outputs/demand/figures/06_recorded_energy.png)

All 200,000 energy values pass basic numeric validity, totaling 9,493,219.61 kWh (47.47 kWh per valid-energy session). Monthly average energy/day ranges from 12,780.06 kWh in September 2025 to 13,319.24 kWh in partial December 2025. Whole session energy is assigned to its start hour/month, including cross-boundary sessions; these are **not measured hourly electricity-load profiles**.

## Monthly energy sensitivity and sample coverage

![Monthly energy sensitivity](../outputs/demand/figures/07_monthly_energy_sensitivity.png)

Battery-capacity concerns flag 49,101 sessions; nominal-charger-energy concerns flag 71,740. Their union is 97,353, with 23,488 flagged by both. The scenario retains 102,647 valid-energy sessions (51.32%) and 3,648,439.11 kWh (38.43% of baseline energy). Baseline session-demand counts are unchanged.

The lower panel shows retained counts; the caption reports date coverage and monthly retention. Exclusions can change sample composition. This scenario is not corrected or ground-truth data, and blank/not-evaluable flags are not confirmation of physical validity.

In [3]:
sensitivity = tables["monthly_energy_sensitivity"]
display(sensitivity.loc[[0, 3, 15, 23], ["Year_Month", "calendar_days", "valid_energy_session_count",
    "retained_valid_energy_session_count", "retained_valid_energy_fraction", "retained_energy_observed_dates"]].round(3))
assert before == {path: file_hash(path) for path in protected}
print("Raw, cleaned, all seven Phase 2 CSVs and analysis_summary.json are unchanged.")

,Year_Month,calendar_days,valid_energy_session_count,retained_valid_energy_session_count,retained_valid_energy_fraction,retained_energy_observed_dates
0,2024-01,31,8468,4373,0.516,31
3,2024-04,30,8100,4173,0.515,30
15,2025-04,30,8365,4222,0.505,30
23,2025-12,30,8434,4343,0.515,30


Raw, cleaned, all seven Phase 2 CSVs and analysis_summary.json are unchanged.


## Limitations and Member 4 handoff

Source origin/licence, collection methodology and timezone remain unverified. Observed timestamps do not prove complete collection. Station IDs have inconsistent location/name mappings; do not make geographic recommendations from them. Session starts do not prove congestion, recorded duration is not necessarily active charging, and true utilization requires reliable capacity/availability information. No causes, significance claims or unsupported recommendations are inferred here.

Member 4 should use cleaned session-level data and a date table for interactive Power BI measures. Use these Python summaries and figures as fixed reference checks under matching filters, not fixed averages reused under incompatible filters. Retain zero-session calendar dates, separate year-months, label partial boundary months and show valid-energy sample coverage. Keep baseline and sensitivity clearly separate. See `docs/demand_methodology.md` and `docs/member2_findings.md`. Station/queue analysis, Power BI integration and the final individual write-up remain outside this step.